# Estado de Resultados Trimestral Pro Forma con PROC COMPUTAB

## Resumen ejecutivo

Este cuaderno construye el estado de resultados trimestral pro forma de un banco regional directamente a partir de datos mensuales del libro mayor usando PROC COMPUTAB, el procedimiento de tablas de redacción de informes de SAS/ETS. Enruta los ingresos por intereses, los gastos por intereses, los ingresos por comisiones y los costos operativos de cada mes a la columna del trimestre calendario correcto, luego usa bloques de programación de filas para calcular el ingreso neto por intereses, el ingreso antes de impuestos y el ingreso neto, y un bloque de columna para consolidar los cuatro trimestres en un total anual.

## Fuentes de datos

El único conjunto de datos sintético `bank_ledger` simula un año fiscal de partidas mensuales del estado financiero para un banco comunitario de tamaño mediano. Doce observaciones mensuales se generan en línea con `call streaminit`/`rand` para que el cuaderno sea completamente autónomo.

| Variable | Tipo | Descripción |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Fecha de cierre mensual del estado (ene–dic) |
| `loanint`  | Num | Intereses y comisiones ganados sobre la cartera de préstamos (miles de USD) |
| `secint`   | Num | Intereses ganados sobre la cartera de valores de inversión (miles de USD) |
| `depint`   | Num | Intereses pagados sobre depósitos (miles de USD) |
| `borrint`  | Num | Intereses pagados sobre préstamos / anticipos del FHLB (miles de USD) |
| `feeinc`   | Num | Ingresos no financieros (comisiones y cargos por servicios) (miles de USD) |
| `salaries` | Num | Gasto de salarios y beneficios a empleados (miles de USD) |
| `occupancy`| Num | Gasto de ocupación y equipo (miles de USD) |
| `othropex` | Num | Otros gastos operativos no financieros (miles de USD) |
| `provision`| Num | Provisión para pérdidas crediticias (miles de USD) |
| `taxrate`  | Num | Tasa impositiva efectiva aplicada al ingreso antes de impuestos |

# Estado de resultados trimestral pro forma con PROC COMPUTAB

Los equipos de finanzas bancarias convierten rutinariamente un libro mayor mensual en un **estado de resultados trimestral** que muestra el ingreso neto por intereses y el ingreso neto final. `PROC COMPUTAB` (SAS/ETS) está diseñado a propósito para esto: dispone una tabla programable cuyas **columnas** son los períodos de reporte y cuyas **filas** son las partidas del estado, y le permite calcular subtotales con lógica ordinaria de paso DATA en bloques de filas y columnas.

En este cuaderno:

1. Generamos un año fiscal de datos mensuales sintéticos del libro mayor para un banco comunitario.
2. Enrutamos cada mes a su trimestre calendario y construimos un estado de resultados trimestral.
3. Calculamos el ingreso neto por intereses, el ingreso antes de impuestos y el ingreso neto en un **bloque de filas**, y consolidamos los trimestres en un total anual en un **bloque de columna**.
4. Reutilizamos la tabla `out=` para que el estado calculado pueda alimentar reportes posteriores.

## Paso 1 — Generar datos mensuales sintéticos del libro mayor

Simulamos doce observaciones de cierre de mes. Los ingresos por intereses de préstamos y valores tienden suavemente al alza durante el año, los costos de depósitos y préstamos escalan con el entorno de tasas, y los ingresos por comisiones, los gastos operativos y la provisión para pérdidas crediticias acarrean ruido realista de mes a mes. `call streaminit` fija la semilla para que los datos sean reproducibles.

In [1]:
DATOS bank_ledger;
   LLAMAR streaminit(20240115);
   FORMATO stmtdate date9.;
   HACER month = 1 HASTA 12;
      /* Fecha de cierre de mes para el año fiscal 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Leve tendencia al alza durante el año + ruido mensual */
      drift = 1 + 0.012 * (month - 1);

      /* Ingresos por intereses (miles de USD) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Gastos por intereses (miles de USD) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Ingresos y gastos no financieros (miles de USD) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provisión para pérdidas crediticias, ocasionalmente elevada */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Tasa impositiva efectiva */
      taxrate = 0.21;

      SALIDA;
   END;
   MANTENER stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=bank_ledger noobs ETIQUETA;
   TÍTULO 'Libro Mayor Mensual Sintético — Banco Comunitario EF2024';
   ETIQUETA stmtdate  = 'Fecha del Estado'
         loanint   = 'Int. Préstamos'
         secint    = 'Int. Valores'
         depint    = 'Int. Depósitos'
         borrint   = 'Int. Financ.'
         feeinc    = 'Ing. Comisiones'
         salaries  = 'Salarios'
         occupancy = 'Ocupación'
         othropex  = 'Otros Gastos'
         provision = 'Provisión'
         taxrate   = 'Tasa Imp.';
   FORMATO loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
EJECUTAR;

                                Libro Mayor Mensual Sintético — Banco Comunitario EF2024                                

Fecha del Estado   Int. Préstamos  Int. Valores   Int. Depósitos  Int. Financ.  Ing. Comisiones  Salarios   Ocupación  Otros Gastos   Provisión  Tasa Imp.
       28JAN2024         1,772.95        423.79           531.47        128.99           339.33    699.38      171.95        202.43      130.93       0.21
       28FEB2024         1,824.38        421.13           564.85        125.53           294.09    767.29      162.79        307.61      123.25       0.21
       28MAR2024         1,931.98        442.06           536.64        131.71           295.72    724.03      153.26        254.21      115.76       0.21
       28APR2024         1,934.99        439.29           535.94        140.01           294.56    729.47      174.19        237.43      198.05       0.21
       28MAY2024         1,949.31        447.35           591.44        124.42           299.78    739.


NOTE: DATA bank_ledger


NOTE: Wrote bank_ledger (12 rows, 11 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=bank_ledger

NOTE: PROC PRINT completed: 12 observations printed, 11 variables


## Paso 2 — Construir el estado de resultados trimestral

El corazón del informe es el paso `PROC COMPUTAB` a continuación.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** define cuatro columnas de trimestre más una columna anual.
- El **bloque de entrada** sin etiqueta establece la variable automática **`_col_`** con `qtr(stmtdate)`, que enruta cada observación mensual a la columna del trimestre correcto. Como la entrada se transpone por defecto, cada variable del libro mayor cae en la fila que comparte su nombre.
- El **bloque de filas** `inc_stmt:` se ejecuta una vez por columna y calcula las líneas derivadas —ingreso neto por intereses, gasto total no financiero, ingreso antes de impuestos e ingreso neto— exactamente como lo haría un contador.
- El **bloque de columna** `total:` se ejecuta una vez por fila y suma los cuatro trimestres en la columna `fy` (anual).

Las sentencias `rows ... / ...` añaden títulos, sangría y líneas de regla (`ol` línea superior, `ul` subrayado, `dul` doble subrayado) para que la salida se lea como un estado financiero real.

In [2]:
TÍTULO 'Estado de Resultados Pro Forma — Community Bancorp, Inc.';
title2 'Año Fiscal 2024  (Importes en miles de USD)';

PROCEDIMIENTO computab DATOS=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Cuatro trimestres más una columna anual consolidada */
   columns qtr1 qtr2 qtr3 qtr4 fy / FORMATO=comma11.1;
   columns qtr1 / 'T1';
   columns qtr2 / 'T2';
   columns qtr3 / 'T3';
   columns qtr4 / 'T4';
   columns fy   / 'Año' 'Completo' +3;

   /* Filas del estado de resultados, de arriba a abajo */
   rows loanint  / 'Intereses y Comisiones de Préstamos';
   rows secint   / 'Intereses sobre Valores'         ul;
   rows intinc   / 'Ingreso Total por Intereses';
   rows depint   / 'Intereses sobre Depósitos';
   rows borrint  / 'Intereses sobre Financiamiento'  ul;
   rows intexp   / 'Gasto Total por Intereses';
   rows nii      / 'Ingreso Neto por Intereses'      ol skip;
   rows provision/ 'Provisión para Pérdidas Cred.'   ul;
   rows niiap    / 'Ing. Neto por Int. tras Provisión' skip;
   rows feeinc   / 'Ingresos No Financieros'         ul;
   rows salaries / 'Salarios y Beneficios';
   rows occupancy/ 'Ocupación y Equipo';
   rows othropex / 'Otros Gastos Operativos'         ul;
   rows nonintexp/ 'Gasto Total No Financiero'       skip;
   rows pretax   / 'Ingreso Antes de Impuestos'      ol;
   rows taxexp   / 'Gasto por Impuesto a la Renta'   ul;
   rows netinc   / 'Ingreso Neto'                    dul;

   /* Bloque de entrada: enrutar cada mes a su trimestre calendario */
   _col_ = qtr(stmtdate);

   /* Bloque de filas: calcular subtotales del estado en cada columna */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Bloque de columna: consolidar los cuatro trimestres en la columna anual */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
EJECUTAR;

                                Estado de Resultados Pro Forma — Community Bancorp, Inc.                                
                                      Año Fiscal 2024  (Importes en miles de USD)                                       


                             The COMPUTAB Procedure                             

                             T1           T2           T3           T4          Año  
                                                                           Completo  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Intereses y Comisiones de Préstamos
  loanint               5,529.3      5,818.7      5,963.5      6,277.0     23,588.4  
  Intereses sobre Valores
  secint                1,287.0      1,334.9      1,342.0      1,452.9      5,416.8  
                    -----------  -----------  -----------  -----------  -----------  
  Ingres


NOTE: Option TITLE changed to Estado de Resultados Pro Forma — Community Bancorp, Inc..
NOTE: Option TITLE2 changed to Año Fiscal 2024  (Importes en miles de USD).
NOTE: PROC COMPUTAB
NOTE: COMPUTAB OUT= dataset written to: qtr_income.csv
NOTE: PROC COMPUTAB statement used.


## Paso 3 — Reutilizar el conjunto de datos de salida de COMPUTAB

El paso `PROC COMPUTAB` anterior escribió su tabla calculada en `qtr_income` mediante `out=`. Cada fila de ese conjunto de datos es una partida del estado y cada variable de columna (`qtr1`–`qtr4`, `fy`) contiene el valor calculado, por lo que puede alimentar reportes posteriores. A continuación imprimimos la columna anual consolidada para confirmar que las cifras cuadran.

In [3]:
TÍTULO 'Conjunto de Datos de Salida de COMPUTAB — Columna Anual';

PROCEDIMIENTO IMPRIMIR DATOS=qtr_income ETIQUETA noobs;
   VAR _row_ fy;
   FORMATO fy comma12.1;
   ETIQUETA _row_ = 'Partida' fy = 'Año Completo';
EJECUTAR;

TÍTULO;

                                Conjunto de Datos de Salida de COMPUTAB — Columna Anual                                 
                                      Año Fiscal 2024  (Importes en miles de USD)                                       


  Partida   Año Completo
---------  -------------
loanint         23,588.4
secint           5,416.8
intinc          29,005.2
depint           6,831.2
borrint          1,650.7
intexp           8,482.0
nii             20,523.2
provision        1,568.9
niiap           18,954.3
feeinc           3,703.2
salaries         8,763.1
occupancy        1,985.6
othropex         2,944.8
nonintexp       13,693.5
pretax           8,964.1
taxexp           1,882.5
netinc           7,081.6




NOTE: Option TITLE changed to Conjunto de Datos de Salida de COMPUTAB — Columna Anual.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Interpretación de los resultados

El estado pro forma se lee de arriba abajo como el informe regulatorio (*call report*) de un banco: el ingreso total por intereses menos el gasto total por intereses produce el **ingreso neto por intereses** —aquí unos \$20.5 millones para el año— el principal motor de ganancias del banco. Restando la **provisión para pérdidas crediticias**, sumando los **ingresos por comisiones** y descontando el **gasto no financiero** se obtiene un ingreso antes de impuestos de aproximadamente \$9.0 millones, y aplicando la tasa impositiva efectiva del 21% se produce un **ingreso neto** cercano a \$7.1 millones. El bloque de columna `total:` suma los cuatro trimestres en la columna anual, por lo que los totales anuales se reconcilian con la suma de los trimestres por construcción —confirmado en el conjunto de datos `out=`, donde el `netinc` anual de 7,081.6 es igual a la suma de las cuatro cifras trimestrales.

Como todo se calcula dentro de los bloques programables de filas y columnas de `PROC COMPUTAB`, sustituir por un libro mayor mensual real no requiere ningún cambio en la lógica del informe —solo cambia el conjunto de datos de entrada. El conjunto de datos `out=` hace entonces que el estado calculado esté disponible para tableros, análisis de tendencias o automatización de paquetes para la junta.